# VDJBet – Yellow Fever vaccination analysis

Replicates the `vdjbet_snippet.Rmd` analysis in Python:

1. Load LLWNGPMAV / HLA-A\*02 TRB entries from VDJdb
2. Generate a large OLGA pool and build 1 000 Pgen-matched mock reference sets
3. For each YFV donor × timepoint, count overlapping clonotypes and cells
4. Z-test against the mock distribution; plot time courses and a heatmap

In [ ]:
import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mir.biomarkers.vdjbet import VDJBetOverlapAnalysis
from mir.common.parser import ClonotypeTableParser, load_vdjdb_latest
from mir.common.repertoire import LocusRepertoire

YFV_DIR = Path("assets/large/yfv19")

## 1  Load VDJdb reference (LLWNGPMAV, HLA-A\*02, TRB)

In [ ]:
vdjdb_rep = load_vdjdb_latest(
    epitope="LLWNGPMAV",
    locus="TRB",
    species="HomoSapiens",
    mhc_a_contains="A*02",
)
print(f"Reference: {vdjdb_rep.clonotype_count} unique clonotypes")
print(f"Example: {vdjdb_rep.clonotypes[0].junction_aa}  {vdjdb_rep.clonotypes[0].v_gene}")

## 2  Create analysis object

`VDJBetOverlapAnalysis` manages the OLGA pool and mock null distributions
internally.  The pool (50 000 TRB sequences with Pgen values) is built
once; mock reference sets are generated lazily and cached on first use.

Two null types are available on demand via `score()` options:
- **pgen-only** (`mock_v_fixed=False, mock_j_fixed=False`) — matches
  the Pgen histogram of the reference, sampling V/J genes randomly.
- **pgen+V+J** (`mock_v_fixed=True, mock_j_fixed=True`) — additionally
  matches V-gene and J-gene usage within each Pgen bin.

In [ ]:
analysis = VDJBetOverlapAnalysis(
    vdjdb_rep,
    n_mocks=1000,
    pool_size=50_000,
    seed=42,
)
print("VDJBetOverlapAnalysis created.")
print("Pool and mocks will be built on the first score() call (cached for subsequent calls).")

## 3  Download YFV dataset from HuggingFace

In [ ]:
if not YFV_DIR.exists() or not any(YFV_DIR.glob("*.airr.tsv.gz")):
    from huggingface_hub import snapshot_download
    print("Downloading isalgo/airr_yfv19 from HuggingFace ...")
    snapshot_download(
        repo_id="isalgo/airr_yfv19",
        repo_type="dataset",
        local_dir=str(YFV_DIR),
    )
    print(f"Downloaded to {YFV_DIR}")
else:
    print(f"Dataset already cached in {YFV_DIR}")

meta = pd.read_csv(YFV_DIR / "metadata.txt", sep="\t")
print(f"{len(meta)} samples:")
meta

## 4  Load YFV samples

In [ ]:
_parser = ClonotypeTableParser()

samples: list[dict] = []
for _, row in meta.iterrows():
    path = YFV_DIR / row["file_name"]
    df = pd.read_csv(path, sep="\t", compression="infer")
    if "locus" in df.columns:
        df = df[df["locus"].fillna("") == "TRB"]
    df = df.dropna(subset=["junction_aa"])
    df = df[df["junction_aa"].str.strip().str.len() > 0]
    clones = _parser.parse_inner(df)
    rep = LocusRepertoire(clonotypes=clones, locus="TRB", repertoire_id=row["file_name"])
    samples.append({
        "file_name": row["file_name"],
        "donor":     row["donor"],
        "day":       int(row["day"]),
        "replica":   str(row.get("replica", "1")),
        "repertoire": rep,
    })

print(f"Loaded {len(samples)} TRB samples")
print(f"Example: {samples[0]['donor']} day {samples[0]['day']}  "
      f"{samples[0]['repertoire'].clonotype_count:,} clonotypes")

## 5  Compute overlaps — exact match and 1mm

`score()` is called once per (sample, options) combination.  The analysis
object caches the OLGA pool and mock null distributions internally, so
repeated calls with different options reuse the same mocks.

| options                                      | Null        | Match  |
|----------------------------------------------|-------------|--------|
| `allow_1mm=False`                            | pgen-only   | exact  |
| `allow_1mm=True`                             | pgen-only   | 1 sub  |
| `mock_v_fixed=True, mock_j_fixed=True`       | pgen+V+J    | exact  |
| `mock_v_fixed=True, mock_j_fixed=True, allow_1mm=True` | pgen+V+J | 1 sub |

Each `OverlapResult` stores `n`, `dc`, `frac_n`, `frac_dc`, `z_n`, `p_n`,
`z_dc`, `p_dc`, plus the raw per-mock arrays for box plots.

In [ ]:
rows: list[dict] = []
for si, s in enumerate(samples):
    rep = s["repertoire"]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        r_pgen     = analysis.score(rep, allow_1mm=False)
        r_pgen_1mm = analysis.score(rep, allow_1mm=True)
        r_pvj      = analysis.score(rep, allow_1mm=False, mock_v_fixed=True, mock_j_fixed=True)
        r_pvj_1mm  = analysis.score(rep, allow_1mm=True,  mock_v_fixed=True, mock_j_fixed=True)

    rows.append({
        "donor":   s["donor"],
        "day":     s["day"],
        "replica": s["replica"],
        # totals
        "n_total":  r_pgen.n_total,
        "dc_total": r_pgen.dc_total,
        # exact / pgen-only
        "n_exact":       r_pgen.n,
        "dc_exact":      r_pgen.dc,
        "frac_n_exact":  r_pgen.frac_n,
        "frac_dc_exact": r_pgen.frac_dc,
        "z_exact_pgen":    r_pgen.z_n,
        "p_exact_pgen":    r_pgen.p_n,
        "z_dc_exact_pgen": r_pgen.z_dc,
        "p_dc_exact_pgen": r_pgen.p_dc,
        # 1mm / pgen-only
        "n_1mm":       r_pgen_1mm.n,
        "dc_1mm":      r_pgen_1mm.dc,
        "frac_n_1mm":  r_pgen_1mm.frac_n,
        "frac_dc_1mm": r_pgen_1mm.frac_dc,
        "z_1mm_pgen":  r_pgen_1mm.z_n,
        "p_1mm_pgen":  r_pgen_1mm.p_n,
        # exact / pgen+V+J
        "z_exact_pvj": r_pvj.z_n,
        "p_exact_pvj": r_pvj.p_n,
        # 1mm / pgen+V+J
        "z_1mm_pvj":   r_pvj_1mm.z_n,
        "p_1mm_pvj":   r_pvj_1mm.p_n,
        # per-mock arrays for box plots
        "mock_pgen_n_exact":       r_pgen.mock_n,
        "mock_pgen_n_1mm":         r_pgen_1mm.mock_n,
        "mock_pvj_n_exact":        r_pvj.mock_n,
        "mock_pvj_n_1mm":          r_pvj_1mm.mock_n,
        "dc_exact_log2":           math.log2(r_pgen.dc + 1),
        "dc_1mm_log2":             math.log2(r_pgen_1mm.dc + 1),
        "mock_pgen_dc_exact_log2": [math.log2(x + 1) for x in r_pgen.mock_dc],
        "mock_pgen_dc_1mm_log2":   [math.log2(x + 1) for x in r_pgen_1mm.mock_dc],
    })
    if (si + 1) % 10 == 0:
        print(f"  {si + 1}/{len(samples)} samples processed")

df_res = pd.DataFrame(rows)
df_res[["donor", "day", "replica", "n_total",
        "n_exact", "n_1mm", "dc_exact", "dc_1mm"]].to_string(index=False)

## 6  Significance stars

z-scores and p-values are already in `df_res` (computed inside `OverlapResult`).
Here we only add human-readable star columns for display in plots and tables.

In [ ]:
def _stars(p) -> str:
    if p is None:  return ""
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return ""


for col, pcol in [
    ("stars_exact_pgen",    "p_exact_pgen"),
    ("stars_exact_pvj",     "p_exact_pvj"),
    ("stars_1mm_pgen",      "p_1mm_pgen"),
    ("stars_1mm_pvj",       "p_1mm_pvj"),
    ("stars_dc_exact_pgen", "p_dc_exact_pgen"),
]:
    df_res[col] = df_res[pcol].apply(_stars)

df_res[["donor", "day", "replica",
        "z_exact_pgen", "stars_exact_pgen",
        "z_exact_pvj",  "stars_exact_pvj",
        "z_1mm_pgen",   "stars_1mm_pgen",
        "z_1mm_pvj",    "stars_1mm_pvj"]]

## 7  Time-course plots

Five panels: exact/1mm × pgen-only/pgen+V+J null, plus duplicate-count (log₂).

In [ ]:
donors   = sorted(df_res["donor"].unique())
replicas = sorted(df_res["replica"].unique())
days_all = sorted(df_res["day"].unique())


def _box_width(days: np.ndarray) -> float:
    if len(days) < 2:
        return 3.0
    gaps = np.diff(np.sort(days))
    return float(gaps[gaps > 0].min()) * 0.4


def _plot_timecourse(real_col: str, mock_col: str, stars_col: str,
                     ylabel: str, title: str) -> None:
    fig, axes = plt.subplots(
        len(replicas), len(donors),
        figsize=(4 * len(donors), 3.5 * len(replicas)),
        squeeze=False,
        sharey=False,
    )
    fig.suptitle(title, fontsize=13, y=1.01)

    for ri, replica in enumerate(replicas):
        for di, donor in enumerate(donors):
            ax = axes[ri][di]
            sub = df_res[
                (df_res["donor"] == donor) & (df_res["replica"] == replica)
            ].sort_values("day")

            if sub.empty:
                ax.set_visible(False)
                continue

            days      = sub["day"].values
            real      = sub[real_col].values
            mock_data = list(sub[mock_col])
            width     = _box_width(days)

            ax.boxplot(
                mock_data,
                positions=days,
                widths=width,
                sym="k.",
                patch_artist=True,
                medianprops=dict(color="#888888", linewidth=1.5),
                boxprops=dict(facecolor="#d0e4f7", alpha=0.8),
                flierprops=dict(markersize=2, markerfacecolor="#888888"),
                manage_ticks=False,
            )

            ax.plot(days, real, "-o", color="#d62728",
                    linewidth=2, markersize=6, zorder=5)

            for day, rv, star in zip(days, real, sub[stars_col]):
                if star:
                    ax.text(day, rv + width * 0.1, star,
                            ha="center", va="bottom",
                            fontsize=11, color="#d62728", fontweight="bold")

            ax.set_title(f"{donor}  R{replica}", fontsize=9)
            ax.set_xlabel("Day post-vaccination", fontsize=8)
            ax.set_ylabel(ylabel, fontsize=8)
            ax.set_xticks(days_all)
            ax.tick_params(labelsize=8)

    plt.tight_layout()
    plt.show()


# Panel 1: exact match, pgen-only null
_plot_timecourse(
    real_col="n_exact",
    mock_col="mock_pgen_n_exact",
    stars_col="stars_exact_pgen",
    ylabel="Matched clonotypes",
    title="LLWNGPMAV – exact match, pgen-only null",
)

# Panel 2: 1mm match, pgen-only null
_plot_timecourse(
    real_col="n_1mm",
    mock_col="mock_pgen_n_1mm",
    stars_col="stars_1mm_pgen",
    ylabel="Matched clonotypes (1mm)",
    title="LLWNGPMAV – 1mm match, pgen-only null",
)

# Panel 3: exact match, pgen+V+J null
_plot_timecourse(
    real_col="n_exact",
    mock_col="mock_pvj_n_exact",
    stars_col="stars_exact_pvj",
    ylabel="Matched clonotypes",
    title="LLWNGPMAV – exact match, pgen+V+J null",
)

# Panel 4: 1mm match, pgen+V+J null
_plot_timecourse(
    real_col="n_1mm",
    mock_col="mock_pvj_n_1mm",
    stars_col="stars_1mm_pvj",
    ylabel="Matched clonotypes (1mm)",
    title="LLWNGPMAV – 1mm match, pgen+V+J null",
)

# Cells (duplicate count) – exact match, pgen-only null (log₂ transformed)
_plot_timecourse(
    real_col="dc_exact_log2",
    mock_col="mock_pgen_dc_exact_log2",
    stars_col="stars_dc_exact_pgen",
    ylabel="log₂(matched cells + 1)",
    title="LLWNGPMAV – exact cells (log₂), pgen-only null",
)

## 8  Heatmap: z-score and significance

In [ ]:
def _heatmap(z_col: str, p_col: str, title: str,
             cmap: str = "RdBu_r", clim: tuple = (-4, 4)) -> None:
    df_res["sample_label"] = df_res["donor"] + "  R" + df_res["replica"]
    pivot_z = df_res.pivot_table(
        index="sample_label", columns="day", values=z_col, aggfunc="first"
    )
    pivot_p = df_res.pivot_table(
        index="sample_label", columns="day", values=p_col, aggfunc="first"
    )

    fig, ax = plt.subplots(figsize=(8, max(4, 0.6 * len(pivot_z))))
    im = ax.imshow(
        pivot_z.values.astype(float),
        aspect="auto", cmap=cmap,
        vmin=clim[0], vmax=clim[1],
        interpolation="nearest",
    )
    plt.colorbar(im, ax=ax, label="z-score", shrink=0.8)

    for ri in range(pivot_z.shape[0]):
        for ci in range(pivot_z.shape[1]):
            p = pivot_p.values[ri, ci]
            z = pivot_z.values[ri, ci]
            if pd.notna(p) and float(p) < 0.05 and pd.notna(z) and float(z) > 0:
                ax.add_patch(plt.Rectangle(
                    (ci - 0.5, ri - 0.5), 1, 1,
                    fill=False, edgecolor="black", linewidth=2,
                ))

    ax.set_xticks(range(len(pivot_z.columns)))
    ax.set_xticklabels([f"Day {d}" for d in pivot_z.columns],
                        rotation=45, ha="right")
    ax.set_yticks(range(len(pivot_z.index)))
    ax.set_yticklabels(pivot_z.index)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


_heatmap("z_exact_pgen", "p_exact_pgen",
         "Clonotype overlap (exact, pgen-only null) – z-score")
_heatmap("z_exact_pvj",  "p_exact_pvj",
         "Clonotype overlap (exact, pgen+V+J null) – z-score")
_heatmap("z_1mm_pgen",   "p_1mm_pgen",
         "Clonotype overlap (1mm, pgen-only null) – z-score")
_heatmap("z_1mm_pvj",    "p_1mm_pvj",
         "Clonotype overlap (1mm, pgen+V+J null) – z-score")
_heatmap("z_dc_exact_pgen", "p_dc_exact_pgen",
         "Cells matched (exact, log₂, pgen-only null) – z-score",
         cmap="Reds", clim=(0, 6))

## 9  Summary table

In [ ]:
summary = (
    df_res[[
        "donor", "day", "replica",
        "n_total", "dc_total",
        "n_exact", "dc_exact",
        "frac_n_exact", "frac_dc_exact",
        "n_1mm", "dc_1mm",
        "frac_n_1mm", "frac_dc_1mm",
        "z_exact_pgen", "p_exact_pgen", "stars_exact_pgen",
        "z_exact_pvj",  "p_exact_pvj",  "stars_exact_pvj",
        "z_1mm_pgen",   "p_1mm_pgen",   "stars_1mm_pgen",
        "z_1mm_pvj",    "p_1mm_pvj",    "stars_1mm_pvj",
        "z_dc_exact_pgen", "p_dc_exact_pgen", "stars_dc_exact_pgen",
    ]]
    .copy()
    .sort_values(["donor", "replica", "day"])
    .reset_index(drop=True)
)

for c in ["frac_n_exact", "frac_dc_exact", "frac_n_1mm", "frac_dc_1mm"]:
    summary[c] = summary[c].map("{:.2e}".format)
for c in ["z_exact_pgen", "z_exact_pvj", "z_1mm_pgen", "z_1mm_pvj", "z_dc_exact_pgen"]:
    summary[c] = summary[c].round(2)
for c in ["p_exact_pgen", "p_exact_pvj", "p_1mm_pgen", "p_1mm_pvj", "p_dc_exact_pgen"]:
    summary[c] = summary[c].map("{:.3f}".format)

summary.rename(columns={
    "n_total":  "clonotypes",
    "dc_total": "cells",
    "n_exact":  "matched_exact",
    "dc_exact": "matched_cells_exact",
    "n_1mm":    "matched_1mm",
    "dc_1mm":   "matched_cells_1mm",
    "frac_n_exact":  "frac_exact",
    "frac_dc_exact": "frac_dc_exact",
    "frac_n_1mm":    "frac_1mm",
    "frac_dc_1mm":   "frac_dc_1mm",
}, inplace=True)

pd.set_option("display.max_rows", 60)
summary